# Task 3: CNN Implementation and Training

This section implements only CNN dataset, transforms, dataloaders, model, and training utilities.


In [ ]:
# Core imports for Task 3
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


In [ ]:
def set_seed(seed: int = 42) -> None:
    """Set random seeds for reproducible training results."""

    # Python random seed
    random.seed(seed)

    # NumPy random seed
    np.random.seed(seed)

    # PyTorch CPU seed
    torch.manual_seed(seed)

    # PyTorch CUDA seeds (only if CUDA exists)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # These flags make cuDNN deterministic for reproducibility.
    # It can be a bit slower, but results are more consistent.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
class SatelliteImageDataset(Dataset):
    """Custom dataset that loads 64x64 RGB satellite images from a DataFrame."""

    def __init__(self, df: pd.DataFrame, transform: transforms.Compose | None = None) -> None:
        """Store dataframe and optional image transform pipeline."""
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        """Return number of rows (images) in the dataframe."""
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Load one image and its numeric label from the dataframe."""

        # Read one row from dataframe
        row = self.df.iloc[idx]

        # Build full path from folder + file_name
        image_path = Path(row["folder"]) / row["file_name"]

        # Load image with PIL and convert to RGB to keep channels consistent
        image = Image.open(image_path).convert("RGB")

        # Apply augmentations/preprocessing if provided
        if self.transform is not None:
            image = self.transform(image)

        # Convert label to long tensor for CrossEntropyLoss
        label = torch.tensor(int(row["label"]), dtype=torch.long)

        return image, label


In [ ]:
def get_transforms() -> tuple[transforms.Compose, transforms.Compose]:
    """Return train and validation transform pipelines."""

    # Same normalization for train and validation
    # Values are common ImageNet-style normalization values
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    # Training transform uses random augmentation
    # This helps reduce overfitting and improves generalization
    train_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    # Validation transform must be deterministic (no random augmentation)
    val_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    return train_transform, val_transform


In [ ]:
def create_dataloaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    batch_size: int = 64,
    num_workers: int = 0
) -> tuple[DataLoader, DataLoader]:
    """Create train and validation dataloaders from dataframes."""

    # Build transform pipelines
    train_transform, val_transform = get_transforms()

    # Build dataset objects
    train_dataset = SatelliteImageDataset(train_df, transform=train_transform)
    val_dataset = SatelliteImageDataset(val_df, transform=val_transform)

    # pin_memory can speed up host->GPU transfer when CUDA is available
    pin_memory = torch.cuda.is_available()

    # Training loader uses shuffle=True for random batch order each epoch
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    # Validation loader uses shuffle=False for stable evaluation
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, val_loader


In [ ]:
class SatelliteCNN(nn.Module):
    """Custom CNN for 64x64 RGB satellite image classification."""

    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()

        # Feature extractor: 4 convolution blocks
        # Each block: Conv -> BatchNorm -> ReLU -> MaxPool
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

        # Adaptive average pooling gives fixed feature size independent of spatial size
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier head with dropout to reduce overfitting
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: features -> pooling -> classifier."""
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


In [ ]:
def calculate_accuracy(outputs: torch.Tensor, labels: torch.Tensor) -> float:
    """Compute batch accuracy from logits and true labels."""

    # Take index of largest logit as predicted class
    preds = outputs.argmax(dim=1)

    # Compare predictions with labels
    correct = (preds == labels).sum().item()
    total = labels.size(0)

    return correct / total if total > 0 else 0.0


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device
) -> tuple[float, float]:
    """Train model for one epoch and return average loss/accuracy."""

    # Enable training mode (important for dropout/batchnorm)
    model.train()

    running_loss = 0.0
    running_acc = 0.0
    num_batches = 0

    for images, labels in dataloader:
        # Move batch to selected device
        images = images.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # CrossEntropyLoss works directly on raw logits
        loss = criterion(outputs, labels)

        # Backward pass computes gradients
        loss.backward()

        # Optimizer updates model weights
        optimizer.step()

        # Track metrics
        running_loss += loss.item()
        running_acc += calculate_accuracy(outputs, labels)
        num_batches += 1

    train_loss = running_loss / max(num_batches, 1)
    train_acc = running_acc / max(num_batches, 1)
    return train_loss, train_acc


def validate_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device
) -> tuple[float, float]:
    """Validate model for one epoch and return average loss/accuracy."""

    # Eval mode disables dropout randomness and uses batchnorm running stats
    model.eval()

    running_loss = 0.0
    running_acc = 0.0
    num_batches = 0

    # no_grad saves memory and speeds up validation because gradients are not needed
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            running_acc += calculate_accuracy(outputs, labels)
            num_batches += 1

    val_loss = running_loss / max(num_batches, 1)
    val_acc = running_acc / max(num_batches, 1)
    return val_loss, val_acc


In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 30,
    learning_rate: float = 1e-3,
    weight_decay: float = 1e-4,
    patience: int = 7,
    model_save_path: str = "assets/weights/best_model.pt"
) -> dict[str, list[float]]:
    """Train the model with early stopping and best-checkpoint saving."""

    # Select GPU if available, else CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # CrossEntropyLoss is standard for multi-class classification with class indices
    criterion = nn.CrossEntropyLoss()

    # AdamW optimizer is a good default for CNN training
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Reduce learning rate when validation accuracy stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )

    # Track metrics across epochs
    history: dict[str, list[float]] = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    # Create weight folder automatically if missing
    save_path = Path(model_save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    best_val_acc = 0.0
    epochs_without_improvement = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # Scheduler uses validation accuracy signal
        scheduler.step(val_acc)

        # Save epoch metrics
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

        # Save best model based on highest validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            epochs_without_improvement = 0
            print(f"  -> Best model updated and saved to {save_path}")
        else:
            epochs_without_improvement += 1

        # Early stopping: stop if no improvement for 'patience' epochs
        if epochs_without_improvement >= patience:
            print("Early stopping triggered.")
            break

    return history


In [ ]:
# Example usage for Task 3
set_seed(42)

train_loader, val_loader = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    batch_size=64,
    num_workers=0,
)

model = SatelliteCNN(num_classes=10)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=30,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=7,
    model_save_path="assets/weights/best_model.pt",
)

print("Training finished.")
print("Best model saved in assets/weights/best_model.pt")
